# Fine-tuning a Pretrained Model
We'll fine-tune `distilbert-base-uncased` for text classification with the 🤗 Trainer API on a tiny subset of the IMDb dataset.
WARNING: This demo purposely trains for ~1 epoch on a small sample for speed; it's about *process*, not accuracy.
Steps:
1. Load dataset
2. Tokenize
3. Define model
4. Train with Trainer
5. Evaluate / inference

In [3]:
!pip install -q transformers torch==2.8.0 pandas==2.2.2 evaluate --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00


In [5]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate
import torch

raw_ds = load_dataset('imdb')
small_train = raw_ds['train'].shuffle(seed=42).select(range(100)) # Reduced from 200
small_test = raw_ds['test'].shuffle(seed=42).select(range(50)) # Reduced from 100

tokenizer = AutoTokenizer.from_pretrained('distilbert/distilbert-base-uncased')

def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True)

train_tok = small_train.map(tokenize_fn, batched=True)
test_tok = small_test.map(tokenize_fn, batched=True)

model = AutoModelForSequenceClassification.from_pretrained('distilbert/distilbert-base-uncased', num_labels=2)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)

args = TrainingArguments(
    output_dir='distilbert-finetuned-imdb-demo',
    eval_strategy='epoch', # Corrected from evaluation_strategy
    per_device_train_batch_size=4, # Reduced from 8
    per_device_eval_batch_size=4, # Reduced from 8
    num_train_epochs=1,
    logging_steps=10,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()
trainer.evaluate()

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-2029895919.py:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: markvoicinovic to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy
1,0.674000,0.673262,0.560000


{'eval_loss': 0.6732617020606995,
 'eval_accuracy': 0.56,
 'eval_runtime': 0.2039,
 'eval_samples_per_second': 245.2,
 'eval_steps_per_second': 63.752,
 'epoch': 1.0}

In [7]:
# inference on new text
text = "This movie was surprisingly touching and beautifully acted."
inputs = tokenizer(text, return_tensors='pt')

# Move inputs to the same device as the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
inputs = {name: tensor.to(device) for name, tensor in inputs.items()}
model.to(device)

with torch.no_grad():
    logits = model(**inputs).logits
pred = torch.argmax(logits, dim=-1).item()
print('Predicted label:', pred, '(1=pos,0=neg)')

Predicted label: 0 (1=pos,0=neg)
